<style>
    body {
        font-family: 'Arial', sans-serif;
        background-color: #f4f4f4;
        color: #333;
        line-height: 1.6;
    }

    .main-container {
        max-width: 940px;
        margin: 20px auto;
        background-color: #ffffff;
        border-radius: 8px;
        box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1);
        padding: 20px;
    }

    h1.title {
        font-size: 3em;
        color: #0056b3; /* Darker blue for a more professional look */
        margin-bottom: 20px;
        font-weight: bold; /* Bold for emphasis */
        text-align: center; /* Center the title */
    }

    h2 {
        font-size: 2em;
        color: #007bff; /* Lighter blue for subheadings */
        margin-top: 40px;
        margin-bottom: 10px;
        font-weight: 600; /* Semi-bold for better visibility */
        border-bottom: 2px solid #e9ecef; /* Underline for emphasis */
        padding-bottom: 5px; /* Space below the heading */
    }

    h3 {
        font-size: 1.75em;
        color: #0056b3;
        margin-top: 20px;
        margin-bottom: 10px;
        font-weight: 500; /* Medium weight for sub-subheadings */
    }

    p {
        font-size: 1.2em;
        margin-bottom: 10px;
        line-height: 1.5; /* Increased line height for better readability */
    }

    pre {
        background-color: #f8f9fa;
        padding: 15px;
        border-radius: 5px;
        overflow-x: auto;
        border: 1px solid #ddd;
    }

    code {
        background-color: #e9ecef;
        padding: 2px 4px;
        border-radius: 4px;
    }

    .dataframe-container {
        overflow-x: auto; /* Enable horizontal scrolling */
        margin-bottom: 20px; /* Space below the DataFrame */
        border: 1px solid #ddd; /* Optional border for better visibility */
        border-radius: 5px; /* Rounded corners */
    }

    table {
        width: 100%; /* Ensure the table takes full width */
        border-collapse: collapse; /* Remove space between cells */
    }

    th, td {
        padding: 10px; /* Padding for table cells */
        text-align: left; /* Align text to the left */
        border-bottom: 1px solid #ddd; /* Light border for separation */
    }

    th {
        background-color: #ffffff; /* Change header background color to white */
        color: #333; /* Header text color */
        font-weight: bold; /* Bold header text */
    }

    tr:hover {
        background-color: #f1f1f1; /* Highlight row on hover */
    }

    .footer {
        text-align: center;
        margin-top: 20px;
        font-size: 0.9em;
        color: #6c757d;
    }

    /* Responsive adjustments */
    @media (max-width: 768px) {
        .main-container {
            padding: 15px;
        }

        h1.title {
            font-size: 2em;
        }

        h2, h3 {
            font-size: 1.5em;
        }
    }
</style>

## Introduction

In this notebook I use LLM from Alibaba called QWEN 2.5 to generate misconceptions based on a prompt I give it (Before reading this notebook I recommend reading first `task_intro_data_preprocess.hmtl` and `similarity_search.html` if you haven't done so already).

The process is done in following steps:
1. First Retrieval: Get top100 best misconceptions by using similarity search as before (using `all_text` (`ConstructName` + `SubjectName` + `QuestionText` + `AnswerText` columns concatenated))
2. Generate misconceptions by prompting QWEN 2.5 and using the top100 misconceptions generated earlier as an example misconceptions in the prompt.
3. Second Retrieval: Get top25 best misconceptions by using similarity search but this time using both `all_text` and QWEN 2.5 output (misconceptions generated by QWEN 2.5).

This method was inspired by the work of takanashihumbert (his work [here](https://www.kaggle.com/code/takanashihumbert/eedi-qwen-2-5-32b-awq-two-time-retrieval#LLM-Reasoning)). My version contains custom fine-tuned embeddings, different prompt message and slightly different hyperparameters of QWEN 2.5 model.

In [ ]:
import os, math, numpy as np
import os
from transformers import AutoTokenizer
import pandas as pd
from tqdm import tqdm
import re, gc
import torch
from sentence_transformers import SentenceTransformer, util
# Display the DataFrame in a scrollable container
from IPython.core.display import display, HTML
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
pd.set_option('display.max_rows', 300)

In [ ]:
# Credit: https://www.kaggle.com/code/abdullahmeda/eedi-map-k-metric

import numpy as np
def apk(actual, predicted, k=25):
    """
    Computes the average precision at k.
    
    This function computes the average prescision at k between two lists of
    items.
    
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
        
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    
    if not actual:
        return 0.0

    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    return score / min(len(actual), k)

def mapk(actual, predicted, k=25):
    """
    Computes the mean average precision at k.
    
    This function computes the mean average prescision at k between two lists
    of lists of items.
    
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
        
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

In [2]:
#IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

df_ret = pd.read_csv("./data/train_df_test.csv")
misconception_mapping = pd.read_csv("./data/misconception_mapping.csv")

## First Retrieval
I get top100 best misconceptions by using mine fine-tuned `SentenceTransformer` to embedd both `all_text` string and `MisconceptionName` and then perform semantic search (same as in previous notebook called `similarity_search`) to retrieve the best 100 relevant misconceptions.

In [ ]:
model = SentenceTransformer('/kaggle/input/eedi-finetuned-bge-public/Eedi-finetuned-bge')

In [3]:
df_ret['all_text'] = df_ret['WholeAssignment'] + '. ' + df_ret['Answer_value']
df['QuestionId_Answer'] = df['QuestionId'].astype(str) + '_' + df['Answer'].str.replace('Answer', '').str.replace('Text', '')

top100_ids = []

embedding_query = model.encode(df['all_text'], convert_to_tensor=True, normalize_embeddings=True)
embedding_Misconception = model.encode(misconception_mapping.MisconceptionName.values, convert_to_tensor=True, normalize_embeddings=True)
top100_ids = util.semantic_search(embedding_query, embedding_Misconception, top_k=100)

df_ret["MisconceptionId"] = [[x["corpus_id"] for x in top100] for top100 in top100_ids]

mapping_dict = misconception_mapping.set_index('MisconceptionId')['MisconceptionName'].to_dict()

df_ret['Retrieval'] = df_ret['MisconceptionId'].apply(lambda ids: [mapping_dict.get(id) for id in ids])

## 2. Generate Misconception using QWEN 2.5

### Create Model Prompts

To generate possible misconceptions I prompt QWEN 2.5 model for each row of the dataset. Where the prompt is in the following format:

"Here is a question about {`ConstructName`}({`SubjectName`}).<br>
Question: {`QuestionText`}<br>
Correct Answer: {`CorrectAnswer`}<br>
Incorrect Answer: {`Answer_value`}<br>

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.<br>
Answer concisely what misconception it is to lead to getting the incorrect answer.<br>
No need to give the reasoning process and do not use "The misconception is" to start your answers.<br>
There are some relative and possible misconceptions below to help you make the decision:

{`Retrival`}"

Here `ConstructName`, `SubjectName`, `QuestionText`, `Answer_value` are values from each of these columns in the dataframe and `Retrival` is the newly retrieved top100 possible misconceptions for this question.

I also tested other different prompts and also using Llama 3.1 model, but the combination of using QWEN 2.5 model with the prompt mentioned above yielded the best results.

In [ ]:
def preprocess_text(x):
    x = re.sub("http\w+", '',x)   # Delete URL
    x = re.sub(r"\.+", ".", x)    # Replace consecutive commas and periods with one comma and period character
    x = re.sub(r"\,+", ",", x)
    x = re.sub(r"\\\(", " ", x)
    x = re.sub(r"\\\)", " ", x)
    x = re.sub(r"[ ]{1,}", " ", x)
    x = x.strip()                 # Remove empty characters at the beginning and end
    return x

PROMPT  = """Here is a question about {ConstructName}({SubjectName}).
Question: {Question}
Correct Answer: {CorrectAnswer}
Incorrect Answer: {IncorrectAnswer}

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.
Answer concisely what misconception it is to lead to getting the incorrect answer.
No need to give the reasoning process and do not use "The misconception is" to start your answers.
There are some relative and possible misconceptions below to help you make the decision:

{Retrival}
"""

PROMPT1  = """Here is a question about {ConstructName}({SubjectName}).
Question: {Question}
Correct Answer: {CorrectAnswer}
Incorrect Answer: {IncorrectAnswer}

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.
Answer concisely what misconception it is to lead to getting the incorrect answer.
No need to give the reasoning process and do not use "The misconception is" to start your answers.
There are some relative and possible misconceptions below to help you make the decision:

{Retrival}

Try to phrase your answer in a similar manner like the possible misconceptions have.
"""

PROMPT2  = """Here is a question about {ConstructName}({SubjectName}).
Question: {Question}
Correct Answer: {CorrectAnswer}
Incorrect Answer: {IncorrectAnswer}
Possible Misconceptions:

{Retrival}

You are a Mathematics teacher. Your task is to identify correct misconception behind the Incorrect Answer with the Question.
Please choose a misconception from the selection of Possible Misconceptions listed above.
Just answer by writing the number of the Possible Misconception.
"""

PROMPT3  = """Here is a question about {ConstructName}({SubjectName}).
Question: {Question}
Correct Answer: {CorrectAnswer}
Incorrect Answer: {IncorrectAnswer}

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.
Answer the question about the misconception concisely in one sentences.
No need to give the reasoning process and describe the misconception without starting with 'The misconception is'..
There are some relative and possible misconceptions below to help you make the decision:

{Retrival}
"""
# just directly give your answers.

def apply_template(row, tokenizer):
    messages = [
        {
            "role": "user", 
            "content": preprocess_text(
                PROMPT.format(
                    ConstructName=row["ConstructName"],
                    SubjectName=row["SubjectName"],
                    Question=row["QuestionText"],
                    IncorrectAnswer=row[f"Answer{targetCol}Text"],
                    CorrectAnswer=row[f"Answer{row.CorrectAnswer}Text"],
                    Retrival=row[f"Retrival"]
                )
            )
        }
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text

df = {}
if not IS_SUBMISSION:
    df_label = {}
    for idx, row in tqdm(df_ret.iterrows(), total=len(df_ret)):
        for option in ["A", "B", "C", "D"]:
            if (row.CorrectAnswer!=option) & (row[f"Misconception{option}Id"]!=-1):
                df[f"{row.QuestionId}_{option}"] = apply_template(row, tokenizer, option)
                df_label[f"{row.QuestionId}_{option}"] = [row[f"Misconception{option}Id"]]
                
    df_label = pd.DataFrame([df_label]).T.reset_index()
    df_label.columns = ["QuestionId_Answer", "MisconceptionId"]
    
    arrays_with_multiple = df_label[df_label['MisconceptionId'].apply(len) > 1]

    if len(arrays_with_multiple) > 0:
        print("Found rows with multiple values:")
        print(arrays_with_multiple)
        print("\nNumber of such rows:", len(arrays_with_multiple))
    else:
        print("All arrays contain exactly one value")
    
    df_label['MisconceptionId'] = df_label['MisconceptionId'].apply(lambda x: x[0])
    df_label['MisconceptionId'] = df_label['MisconceptionId'].astype(int)

    df_label = df_label.merge(
        df_misconception_mapping[['MisconceptionId', 'MisconceptionName']], 
        on='MisconceptionId', 
        how='left'
    )
    df_label.to_parquet("label.parquet", index=False)
else:
    for idx, row in tqdm(df_ret.iterrows(), total=len(df_ret)):
        for option in ["A", "B", "C", "D"]:
            if row.CorrectAnswer!=option:
                df[f"{row.QuestionId}_{option}"] = apply_template(row, tokenizer, option)

df = pd.DataFrame([df]).T.reset_index()
df.columns = ["QuestionId_Answer", "text"]
df.to_parquet("submission.parquet", index=False)

So, an example prompt looks the following:

In [9]:
print(df.loc[0, 'text'])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Here is a question about Divide two decimals with the same number of decimal places(Multiplying and Dividing with Decimals).
Question: 0.9 \div 0.3= 
Correct Answer: 3 
Incorrect Answer: 0.3 

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.
Answer concisely what misconception it is to lead to getting the incorrect answer.
No need to give the reasoning process and do not use "The misconception is" to start your answers.
There are some relative and possible misconceptions below to help you make the decision:

1. When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places
2. Subtracts instead of divides
3. Divides rather than multiplies 
4. Divides rather than multiplies in a related calculation questions
5. Multipl

### Generate responses

After the prompts have been created for all questions, they are then fed to QWEN 2.5 model to generate responses.

In [ ]:
%%writefile run_vllm.py

import re
import vllm
import pandas as pd

df = pd.read_parquet("submission.parquet")

model_path = "/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1"

llm = vllm.LLM(
    model_path,
    quantization="awq",
    tensor_parallel_size=2,
    gpu_memory_utilization=0.90, 
    trust_remote_code=True,
    dtype="half", 
    enforce_eager=True,
    max_model_len=5120,
    disable_log_stats=True
)
tokenizer = llm.get_tokenizer()


responses = llm.generate(
    df["text"].values,
    vllm.SamplingParams(
        n=1,
        top_p=0.8, 
        temperature=0.5, 
        seed=777, 
        skip_special_tokens=False,  
        max_tokens=1024, 
    ),
    use_tqdm=True
)

responses = [x.outputs[0].text for x in responses]
df["fullLLMText"] = responses

def extract_response(text):
    return ",".join(re.findall(r"<response>(.*?)</response>", text)).strip()

df["llmMisconception"] = responses
df.to_parquet("submission.parquet", index=False)

In [ ]:
!python run_vllm.py

After running the model the responses look the following:

In [12]:
llm_output = pd.read_parquet("submission.parquet")
for idx, row in llm_output[0:5].iterrows():
    print(row.llmMisconception)
    print("==="*6)

When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places.
When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places
Believes squaring is the same as adding two
5
Believes rounding must change the number


And when I compare the first ten responses to the true misconceptions (as seen below) its visible that the LLM's output seems to be close on number of occasions. However, I can also see that the model sometimes outputs just a single number. This number is presumably the example number from the 100 potential misconception was given to it in the prompt. 

In [13]:
llm_output = pd.read_parquet("submission.parquet")
true_output = pd.read_parquet("label.parquet")
for idx, (llm_row, true_row) in enumerate(zip(llm_output[:10].itertuples(), 
                                                 true_output[:10].itertuples())):
        print("LLM Output:")
        print(llm_row.llmMisconception)
        print("\nTrue Output:")
        print(true_row.MisconceptionName)
        print('=' * 18 + '\n')

LLM Output:
When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places.

True Output:
When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places

LLM Output:
When dividing decimals with the same number of decimal places as each other, assumes the answer also has the same number of decimal places

True Output:
Believes division is commutative 

LLM Output:
Believes squaring is the same as adding two

True Output:
Mixes up squaring and addition

LLM Output:
5

True Output:
Mixes up squaring and multiplying by 2 or doubling

LLM Output:
Believes rounding must change the number

True Output:
Rounds to the wrong degree of accuracy (rounds too much)

LLM Output:
Rounds to the wrong degree of accuracy (rounds too much)

True Output:
Rounds up instead of down

LLM Output:
Changes wrong place value column when rounding

True Output:
Roun

### Post-processing LLM's output

As mentioned above QWEN 2.5 sometimes outputs just a single number. This number is therefore mapped to the corresponding misconception the LLM picked as the most fitting one.

In [16]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

df = pd.read_parquet("submission.parquet")
df_misconception_mapping = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/misconception_mapping.csv")

model = SentenceTransformer('/kaggle/input/eedi-finetuned-bge-public/Eedi-finetuned-bge')

In [17]:
def number2sentence(row):
    """
    This is used for post-processing of LLM's output.
    Since we give top-N retrieval to the LLM with serial number,
    Sometimes the LLM will only output the serial number without any sentence.
    We use the 'dicts' generated at the beginning to map the serial number with corresponding misconceptions.
    """
    text = row['llmMisconception'].strip()    
    contains_number = re.search(r'^(\d+)', text)
    
    if contains_number:
        potential = contains_number.group(1)
        qid_retrieval = dicts[row['QuestionId']]
        try:
            # qid_retrieval is the top-N misconceptions for an QuestionId,
            # qid_retrieval[potential] is the most possible misconception.
            text = qid_retrieval[potential]
        except:
            # If the mapping fails, we use the first one(the most possible one in the first retrieval).
            text = qid_retrieval['1']
        
    return text


df['QuestionId'] = df['QuestionId_Answer'].apply(lambda x: x.split('_')[0])
df['llmMisconception_clean'] = df.apply(number2sentence, axis=1)

In the dataset below in column called `llmMisconception` are the original LLM's responses and in column `llmMisconception_clean` are the responses after the aforementioned mapping.

In [18]:
df.head(5)

,QuestionId_Answer,text,fullLLMText,llmMisconception,QuestionId,llmMisconception_clean
0,1700_A,"<|im_start|>system\nYou are Qwen, created by A...",When dividing decimals with the same number of...,When dividing decimals with the same number of...,1700,When dividing decimals with the same number of...
1,1700_C,"<|im_start|>system\nYou are Qwen, created by A...",When dividing decimals with the same number of...,When dividing decimals with the same number of...,1700,When dividing decimals with the same number of...
2,1488_A,"<|im_start|>system\nYou are Qwen, created by A...",Believes squaring is the same as adding two,Believes squaring is the same as adding two,1488,Believes squaring is the same as adding two
3,1488_B,"<|im_start|>system\nYou are Qwen, created by A...",5,5,1488,Mixes up squaring and multiplying by 2 or doub...
4,921_B,"<|im_start|>system\nYou are Qwen, created by A...",Believes rounding must change the number,Believes rounding must change the number,921,Believes rounding must change the number


## 3. Second Retrieval

Now to each generated misconception, I wanna pick the top25 closest misconception from the `misconception_mapping` dataset. These top25 misconceptions for each question will be then the resulting ones that I submit.

To do this I use semantic search again. Firstly both `misconception_mapping` dataset, and the generated misconception concatenated with the its prompt are encoded (concatenating the generated misconception with the prompt yielded better results). For encoding I use again fine-tuned version of bge-large-1.5 (again fine-tuned in the same manner as the one for the similarity search).

After that, semantic search is applied to get the best 25 misconceptions for each prompt+generated misconception input. 

In [ ]:
def preprocess_text(x):
    x = x.lower()                 # Convert words to lowercase
    x = re.sub(r"@\w+", '',x)      # Delete strings starting with @
    #x = re.sub(r"\d+", '',x)      # Delete Numbers
    x = re.sub(r"http\w+", '',x)   # Delete URL
    x = re.sub(r"\\\(", " ", x)
    x = re.sub(r"\\\)", " ", x)
    x = re.sub(r"[ ]{1,}", " ", x)
    x = re.sub(r"\.+", ".", x)    # Replace consecutive commas and periods with one comma and period character
    x = x.strip()                 # Remove empty characters at the beginning and end
    return x

PREFIX = "<|im_start|>user"
df['input_features'] = df["text"].apply(lambda x: x.split(PREFIX)[1].split("You are a Mathematics teacher.")[0].strip('\n').split('Here is a question about')[-1].strip())

df['input_features'] = df['input_features'].apply(lambda x: preprocess_text(x))
df['input_features'] = df["llmMisconception_clean"] + "\n\n" + df['input_features']

embedding_query = model.encode(df['input_features'], convert_to_tensor=True, normalize_embeddings=True)
embedding_Misconception = model.encode(df_misconception_mapping.MisconceptionName.values, convert_to_tensor=True, normalize_embeddings=True)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
SEMANTIC_SEARCH = True
top25ids = []

if SEMANTIC_SEARCH:
    embedding_query = model.encode(df['input_features'], convert_to_tensor=True, normalize_embeddings=True)
    embedding_Misconception = model.encode(df_misconception_mapping.MisconceptionName.values, convert_to_tensor=True, normalize_embeddings=True)
    top25ids = util.semantic_search(embedding_query, embedding_Misconception, top_k=25)
    
else: #cosine similarity
    embedding_query = model.encode(df['input_features'], convert_to_tensor=False, normalize_embeddings=True)
    embedding_Misconception = model.encode(df_misconception_mapping.MisconceptionName.values, convert_to_tensor=False, normalize_embeddings=True)
    similarities = cosine_similarity(embedding_query, embedding_Misconception)
    # Get top 25 indices for each query
    for sim_row in similarities:
        top_indices = np.argsort(sim_row)[-25:][::-1]  # Sort in descending order
        top_matches = [{'corpus_id': int(idx), 'score': float(sim_row[idx])} for idx in top_indices]
        top25ids.append(top_matches)

In the following dataset under `MisconceptionId` the top 25 misconceptions (their Ids rather) for each question are seen.

In [21]:
df["MisconceptionId"] = [" ".join([str(x["corpus_id"]) for x in top25id]) for top25id in top25ids]

df[["QuestionId_Answer", "MisconceptionId"]].to_csv("submission.csv", index=False)
display(HTML(f'<div class="dataframe-container">{df.head().to_html(index=False)}</div>'))

,QuestionId_Answer,text,fullLLMText,llmMisconception,QuestionId,llmMisconception_clean,input_features,MisconceptionId
0,1700_A,"<|im_start|>system\nYou are Qwen, created by A...",When dividing decimals with the same number of...,When dividing decimals with the same number of...,1700,When dividing decimals with the same number of...,When dividing decimals with the same number of...,153 2273 339 2359 447 151 1259 2542 565 2525 1...
1,1700_C,"<|im_start|>system\nYou are Qwen, created by A...",When dividing decimals with the same number of...,When dividing decimals with the same number of...,1700,When dividing decimals with the same number of...,When dividing decimals with the same number of...,153 2273 339 151 2359 447 2542 376 1259 565 15...
2,1488_A,"<|im_start|>system\nYou are Qwen, created by A...",Believes squaring is the same as adding two,Believes squaring is the same as adding two,1488,Believes squaring is the same as adding two,Believes squaring is the same as adding two\n\...,948 729 2352 952 278 1072 2445 2316 1664 1137 ...
3,1488_B,"<|im_start|>system\nYou are Qwen, created by A...",5,5,1488,Mixes up squaring and multiplying by 2 or doub...,Mixes up squaring and multiplying by 2 or doub...,2316 1137 948 902 278 1664 2175 1072 64 2352 7...
4,921_B,"<|im_start|>system\nYou are Qwen, created by A...",Believes rounding must change the number,Believes rounding must change the number,921,Believes rounding must change the number,Believes rounding must change the number\n\nro...,322 822 1402 1861 1675 2412 1605 2162 1744 135...


## Results
When I run this whole notebook on my test dataset and compare my results to the ground truth I get the following result.

In [22]:
if not IS_SUBMISSION:
    import pandas as pd
    from eedi_metrics import mapk
    predicted = pd.read_csv("submission.csv")["MisconceptionId"].apply(lambda x: [int(y) for y in x.split()])
    label = pd.read_parquet("label.parquet")["MisconceptionId"].apply(lambda x: [x])
    print("Validation: ", mapk(label, predicted))

Validation:  0.3877271655216676


## Evaluation and Summary

A MAP@25 score of 0.39 is a much better then score of 0.21 which was a score of using similarity search only. On the Kaggle competition test set my models achieved scores of 0.37 and 0.23 respectively. These results are far from perfect and placed me in top300 competitors out of 1449 people (for reference the best score achieved was 0.63).

After the competition ended the the top competitors shared their solutions, I could compare their approaches to mine. All of the top competitors also used LLMs to generate misconception, however they either fully trained the LLMs or they used LoRA. This approach is unfortunately computationally imposable due to my lack of resources (I fine-tuned models using Kaggle's online notebook, since this is where I found the biggest free RAM and GPU sizes).

However, a lot of competitors used different ways of how they then retrieved the top25 misconceptions (from using different embedding model in similarity search, to different training methods of the embedding models, to not using similarity search at all and instead relying again on LLMs to pick the top25 misconceptions). These are the methods I could have explored even with my limited computational resources.

So in summary, there would be way to probably improve on my models performance. And although I haven't reached the a great spot on competitions leaderboard I still enjoyed tackling this problem and learning things along the way.
